# MultiModal-DVRP-RL — demo

A guided walkthrough of learning a **dispatch policy** against the
[MOSAIC](https://github.com/smarttransit-ai/MOSAIC) DVRP simulator. MOSAIC is the
*environment*; this repo owns the learning code.

**Prereqs:** MOSAIC installed (`pip install -e ".[dev,notebook]"`; see the README
for private-repo access). The **first env build downloads + caches the OSM road
graph** under `cache/` (needs network once, ~1–2 min); later runs reuse it.

Sections: **1** build the env · **2** one episode · **3** baselines ·
**4** the `Rollout` planner (fast demo). For the full benchmark numbers see
[`RESULTS.md`](../RESULTS.md).

## 0. Setup

Resolve the repo root so config paths work whether this notebook is launched
from the repo root or from `notebooks/`.

In [1]:
from pathlib import Path

# Walk up to the repo root (the dir containing pyproject.toml).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = ROOT / "configs" / "binghampton.yaml"
print("repo root:", ROOT)
print("config:   ", CONFIG.name)

repo root: /Users/jose/Developer/git/MultiModal-DVRP-RL
config:    binghampton.yaml


## 1. Build the environment from a config

`make_env_from_config` is our one entry point: it reads the YAML config, builds
the MOSAIC scenario spec, and returns a ready-to-step `(env, policy)`. The first
call fetches + caches the OSM graph.

In [2]:
from dvrp_rl.env import make_env_from_config

# policy="on_demand_only" is MOSAIC's stock accept-everything policy.
env, policy = make_env_from_config(CONFIG, policy="on_demand_only", seed=42)
state = env.reset()
print(f"env ready · vehicles: {len(state.vehicles)} · t0: {state.current_time:.1f}s")

Trimmed graph to largest SCC: 501 -> 496 nodes (99% kept)


env ready · vehicles: 3 · t0: 66.7s


## 2. Run one episode

Drive the env: the policy decides on each incoming request, `env.step` applies it
and advances to the next request. `drain()` finishes serving accepted trips, then
we read the service rate (served / total requests).

In [3]:
N_STEPS = 200
env, policy = make_env_from_config(CONFIG, policy="on_demand_only", seed=42)
state = env.reset()
for _ in range(N_STEPS):
    action = policy.create_trips(state)          # a Trip (accept) or None (reject)
    state, _reward, done, _info = env.step(action)
    if done:
        break
env.drain()
m = env.metrics
print(f"served {m.served_requests}/{m.total_requests} -> service_rate {m.service_rate:.1%}")
env.close()

served 169/200 -> service_rate 84.5%


## 3. Baselines

`evaluate` runs a policy across several seeds and reports mean service rate.
`AcceptAll` accepts everything (the number to beat); `Random` flips a coin. We
use a couple of seeds + short episodes here to keep the demo quick — see
`RESULTS.md` for the full runs.

In [4]:
from dvrp_rl.evaluate import evaluate
from dvrp_rl.policies import AcceptAll, RandomPolicy

SEEDS = [100, 101]
for name, make in [("AcceptAll", lambda s: AcceptAll()),
                   ("Random(0.5)", lambda s: RandomPolicy(accept_prob=0.5, seed=s))]:
    r = evaluate(CONFIG, make, SEEDS, n_steps=150)
    print(f"{name:<12} service_rate: {r['mean_service_rate']:.1%}")

AcceptAll    service_rate: 93.0%


Random(0.5)  service_rate: 50.3%


## 4. The `Rollout` planner (fast demo)

`RolloutPolicy` does a one-step lookahead: for each request it clones the env and
simulates *accept* vs *reject*, then picks the branch that serves more. Two modes:

- **oracle** — the clone keeps the true demand RNG, so it plans with *perfect
  foresight* of future requests. An **upper bound**, not deployable.
- **sampled** (default) — reseeds each clone's demand RNG, planning against
  *sampled* futures (never the real one), averaged over `n_samples`. Honest, but
  costs K clones per decision.

> Settings below are tiny (`horizon=5`, few steps) **for a quick demo** — this is
> not a benchmark. The real, slower runs are in `RESULTS.md`.

In [5]:
from dvrp_rl.policies import RolloutPolicy

# oracle (perfect foresight) — fast demo settings
r = evaluate(CONFIG, lambda s: RolloutPolicy(horizon=5, oracle=True), [100], n_steps=60)
print(f"Rollout(oracle, H=5, demo)      service_rate: {r['mean_service_rate']:.1%}")

# sampled (honest, no foresight) — reseeds demand per rollout; also tiny here
r = evaluate(CONFIG, lambda s: RolloutPolicy(horizon=5, n_samples=2, sample_seed=s), [100], n_steps=60)
print(f"Rollout(sampled, H=5, K=2, demo) service_rate: {r['mean_service_rate']:.1%}")

Rollout(oracle, H=5, demo)      service_rate: 96.7%


Rollout(sampled, H=5, K=2, demo) service_rate: 96.7%


## Where next

- Full benchmark numbers (all seeds, `horizon=30`): [`RESULTS.md`](../RESULTS.md).
  Oracle beats `AcceptAll` (+3.1pp, +7.2pp congested); the honest sampled planner
  trails at low sample counts and improves with more samples.
- **Next milestone — REINFORCE:** a *learned* policy over the geography-free
  feature vector (`dvrp_rl.features`) that decides from the current state only —
  deployable and fast, amortizing what the sampled rollout does expensively
  per-decision. See [`plan.md`](../plan.md).